In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
!gdown 1nc7YEa9i0xlsTIPLQEjWPLgikOoigxjL

Downloading...
From: https://drive.google.com/uc?id=1nc7YEa9i0xlsTIPLQEjWPLgikOoigxjL
To: /content/novi_final_20k.csv
100% 84.4M/84.4M [00:01<00:00, 52.4MB/s]


In [4]:
df = pd.read_csv("novi_final_20k.csv")

In [5]:


TEXT_COL = "SADRZAJ"   # pazi da nije "SADRZAj" ili "SADRŽAJ"

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

KEYWORDS = {
    "euroatlantske_integracije": [
        "nato", "eu", "evropska unija", "europska unija", "euroatlantske integracije",
        "evroatlantske integracije", "članstvo u nato", "pristupanje eu",
        "kandidatski status", "brisel", "integracije u eu", "integracije u nato",
        "atlantski savez", "savez za mir", "euroatlantski put", "evropski put"
    ],

    "negiranje_genocida": [
        "genocid", "srebrenica", "negiranje genocida", "nije bio genocid",
        "rezolucija o srebrenici", "ratni zločin", "ratni zlocin",
        "presuda za genocid", "haški tribunal", "haski tribunal",
        "icty", "međunarodni sud", "medjunarodni sud"
    ],

    "gradjanska_vs_konstitutivni": [
        "građanska država", "gradjanska država", "građanski model", "gradjanski model",
        "konstitutivni narodi", "konstitutivnost", "legitimno predstavljanje",
        "jedan čovjek jedan glas", "jedan covjek jedan glas",
        "majorizacija", "vitalni nacionalni interes",
        "bošnjaci hrvati srbi", "bosnjaci hrvati srbi"
    ],

    "izborna_reforma": [
        "izborni zakon", "izborna reforma", "izmjene izbornog zakona",
        "centralna izborna komisija", "cik", "integritet izbora",
        "elektronsko glasanje", "skeneri", "krađa izbora", "kradja izbora",
        "birački odbori", "biracki odbori", "izborni proces",
        "kompenzacijski mandati", "dom naroda", "predsjedništvo bih", "predsjednistvo bih"
    ],
}


def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace("đ", "dj")
    x = x.replace("č", "c").replace("ć", "c")
    x = x.replace("š", "s").replace("ž", "z")
    return x


def normalize_keyword(x):
    x = x.lower()
    x = x.replace("đ", "dj")
    x = x.replace("č", "c").replace("ć", "c")
    x = x.replace("š", "s").replace("ž", "z")
    return x


KEYWORDS_NORM = {
    topic: [normalize_keyword(k) for k in kws]
    for topic, kws in KEYWORDS.items()
}


def count_keyword_matches(text, keywords):
    matches = []
    score = 0

    for kw in keywords:
        pattern = r"(?<!\w)" + re.escape(kw) + r"(?!\w)"
        found = re.findall(pattern, text)

        if found:
            matches.append(kw)
            score += len(found)

    return score, matches


def add_keyword_topic_columns(df, min_hits=1):
    out = df.copy()

    normalized_texts = out[TEXT_COL].fillna("").map(normalize_text)

    for topic in TOPICS:
        scores = []
        matched = []

        for text in normalized_texts:
            score, matches = count_keyword_matches(text, KEYWORDS_NORM[topic])
            scores.append(score)
            matched.append(matches)

        out[f"{topic}_score"] = scores
        out[f"{topic}_matched"] = matched
        out[f"{topic}_mentioned"] = (out[f"{topic}_score"] >= min_hits).astype(int)

    mentioned_cols = [f"{t}_mentioned" for t in TOPICS]

    out["not_mentioned"] = (out[mentioned_cols].sum(axis=1) == 0).astype(int)

    score_cols = [f"{t}_score" for t in TOPICS]
    out["primary_topic"] = out[score_cols].idxmax(axis=1).str.replace("_score", "", regex=False)
    out.loc[out["not_mentioned"] == 1, "primary_topic"] = "not_mentioned"

    return out


df_kw = add_keyword_topic_columns(df, min_hits=1)

df_kw[
    [TEXT_COL, "primary_topic", "not_mentioned"] +
    [f"{t}_mentioned" for t in TOPICS] +
    [f"{t}_score" for t in TOPICS]
].head()

,SADRZAJ,primary_topic,not_mentioned,euroatlantske_integracije_mentioned,negiranje_genocida_mentioned,gradjanska_vs_konstitutivni_mentioned,izborna_reforma_mentioned,euroatlantske_integracije_score,negiranje_genocida_score,gradjanska_vs_konstitutivni_score,izborna_reforma_score
0,Kantoni u Federaciji BiH zaključno sa 31. dece...,not_mentioned,1,0,0,0,0,0,0,0,0
1,Na poziv članica Međunarodne asocijacije vozač...,euroatlantske_integracije,0,1,0,0,0,2,0,0,0
2,Hermin Zornić i Slaven Zeljko smijenjeni su sa...,not_mentioned,1,0,0,0,0,0,0,0,0
3,Dan nakon 11. jula kolektivne dženaze i ukopa ...,negiranje_genocida,0,0,1,0,0,0,4,0,0
4,"U pismu dugačkom tri stranice, u koje je uvid ...",euroatlantske_integracije,0,1,0,0,0,2,0,0,0


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TOPIC_DESCRIPTIONS = {
    "euroatlantske_integracije": """
        NATO Evropska unija EU euroatlantske integracije evropski put članstvo u NATO
        pristupanje Evropskoj uniji Brisel kandidatski status reformski put
    """,

    "negiranje_genocida": """
        genocid Srebrenica negiranje genocida ratni zločini presude međunarodni sud
        haški tribunal rezolucija o Srebrenici veličanje ratnih zločinaca
    """,

    "gradjanska_vs_konstitutivni": """
        građanska država konstitutivni narodi legitimno predstavljanje majorizacija
        jedan čovjek jedan glas Dom naroda vitalni nacionalni interes uređenje BiH
    """,

    "izborna_reforma": """
        izborna reforma izborni zakon CIK centralna izborna komisija integritet izbora
        elektronsko glasanje birački odbori krađa izbora izmjene izbornog zakona
    """,
}


def add_tfidf_topic_columns(df, threshold=0.10):
    out = df.copy()

    texts = out[TEXT_COL].fillna("").map(normalize_text).tolist()
    topic_texts = [normalize_text(TOPIC_DESCRIPTIONS[t]) for t in TOPICS]

    all_texts = texts + topic_texts

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.90,
        sublinear_tf=True
    )

    X = vectorizer.fit_transform(all_texts)

    X_docs = X[:len(texts)]
    X_topics = X[len(texts):]

    sims = cosine_similarity(X_docs, X_topics)

    for i, topic in enumerate(TOPICS):
        out[f"{topic}_tfidf_score"] = sims[:, i]
        out[f"{topic}_mentioned"] = (out[f"{topic}_tfidf_score"] >= threshold).astype(int)

    mentioned_cols = [f"{t}_mentioned" for t in TOPICS]
    out["not_mentioned"] = (out[mentioned_cols].sum(axis=1) == 0).astype(int)

    score_cols = [f"{t}_tfidf_score" for t in TOPICS]
    out["primary_topic"] = out[score_cols].idxmax(axis=1).str.replace("_tfidf_score", "", regex=False)
    out.loc[out["not_mentioned"] == 1, "primary_topic"] = "not_mentioned"

    return out


df_tfidf = add_tfidf_topic_columns(df, threshold=0.10)

df_tfidf[
    [TEXT_COL, "primary_topic", "not_mentioned"] +
    [f"{t}_mentioned" for t in TOPICS] +
    [f"{t}_tfidf_score" for t in TOPICS]
].head()

,SADRZAJ,primary_topic,not_mentioned,euroatlantske_integracije_mentioned,negiranje_genocida_mentioned,gradjanska_vs_konstitutivni_mentioned,izborna_reforma_mentioned,euroatlantske_integracije_tfidf_score,negiranje_genocida_tfidf_score,gradjanska_vs_konstitutivni_tfidf_score,izborna_reforma_tfidf_score
0,Kantoni u Federaciji BiH zaključno sa 31. dece...,not_mentioned,1,0,0,0,0,0.000000,0.000000,0.000321,0.000000
1,Na poziv članica Međunarodne asocijacije vozač...,not_mentioned,1,0,0,0,0,0.001773,0.000000,0.001227,0.002549
2,Hermin Zornić i Slaven Zeljko smijenjeni su sa...,not_mentioned,1,0,0,0,0,0.000000,0.000000,0.001068,0.000000
3,Dan nakon 11. jula kolektivne dženaze i ukopa ...,not_mentioned,1,0,0,0,0,0.000000,0.014000,0.002056,0.000000
4,"U pismu dugačkom tri stranice, u koje je uvid ...",not_mentioned,1,0,0,0,0,0.003206,0.003081,0.000000,0.000000


# Kemans

In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


TEXT_COL = "SADRZAJ"

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
    "not_mentioned"
]


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace("đ", "dj")
    x = x.replace("č", "c").replace("ć", "c")
    x = x.replace("š", "s").replace("ž", "z")
    return x


df_cluster = df.copy()
df_cluster["text_clean"] = df_cluster[TEXT_COL].fillna("").map(clean_text)

vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True
)

X = vectorizer.fit_transform(df_cluster["text_clean"])

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

df_cluster["cluster"] = kmeans.fit_predict(X)

print(df_cluster["cluster"].value_counts())

cluster
1    7248
2    4977
4    4237
0    3214
3     144
Name: count, dtype: int64


# sentence

In [8]:
!pip install sentence-transformers -q

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = df[TEXT_COL].fillna("").astype(str).tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

kmeans_emb = KMeans(
    n_clusters=15,
    random_state=42,
    n_init=10
)

df_emb_cluster = df.copy()
df_emb_cluster["cluster"] = kmeans_emb.fit_predict(embeddings)

print(df_emb_cluster["cluster"].value_counts())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/310 [00:00<?, ?it/s]

cluster
7     2079
2     2002
12    1923
4     1554
9     1489
11    1463
10    1462
6     1306
3     1300
14    1238
13    1163
8      983
1      953
5      646
0      259
Name: count, dtype: int64


In [10]:
kmeans_emb = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

df_emb_cluster = df.copy()
df_emb_cluster["cluster"] = kmeans_emb.fit_predict(embeddings)

print(df_emb_cluster["cluster"].value_counts())

cluster
3    5407
4    5104
0    4317
1    3845
2    1147
Name: count, dtype: int64


In [11]:
df_cluster[df_cluster["cluster"]==1][["naslov","SADRZAJ","cluster"]].head()

,naslov,SADRZAJ,cluster
0,"Krediti se dižu, novca nema: Kanton Sarajevo d...",Kantoni u Federaciji BiH zaključno sa 31. dece...,1
1,Stanje tržišta električnih vozila u BiH predst...,Na poziv članica Međunarodne asocijacije vozač...,1
2,"Dvojac u BiH prvo uhapšen, pa sada ostao bez v...",Hermin Zornić i Slaven Zeljko smijenjeni su sa...,1
7,Vlada FBiH odobrila 200.000 KM za obnovu Parti...,Kako je federalni premijer Fadil Novalić i naj...,1
8,Objavljeni satelitski snimci razaranja u Libij...,"Više od 5.000 osoba izgubilo je život, dok se ...",1


In [12]:
df_emb_cluster[df_emb_cluster["cluster"]==1][["naslov"]].head(15)

,naslov
0,"Krediti se dižu, novca nema: Kanton Sarajevo d..."
1,Stanje tržišta električnih vozila u BiH predst...
8,Objavljeni satelitski snimci razaranja u Libij...
11,Danas Tucindan i praznik Oci: Šta kažu običaji?
14,Misterija stara deceniju: Nastavlja se potraga...
21,Ćeranić: Tramp će se sa Gejtsom i Sorošem obra...
22,Biljni svijet je dehidrirao u nedostatku vlage...
25,Bh. kardiolog Midhat Nurkić poslao važnu poruk...
34,Najneiskreniji govor u Trumpovoj historiji: Sv...
40,Sud u Hagu donio novu presudu protiv Izraela: ...


In [13]:
import pandas as pd
import numpy as np

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

df_eval = df.copy()

def to_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return x == 1
    if isinstance(x, str):
        return x.strip().lower() in ["true", "1", "yes", "da"]
    return False

for topic in TOPICS:
    col = f"{topic}_mentioned"
    df_eval[col] = df_eval[col].apply(to_bool)


def get_true_topic(row):
    mentioned = [
        topic for topic in TOPICS
        if row[f"{topic}_mentioned"] == True
    ]

    if len(mentioned) == 0:
        return "not_mentioned"

    if len(mentioned) == 1:
        return mentioned[0]

    return "multi_topic"


df_eval["true_topic"] = df_eval.apply(get_true_topic, axis=1)

df_eval["true_topic"].value_counts()

,count
true_topic,
not_mentioned,7603
multi_topic,4176
euroatlantske_integracije,3206
negiranje_genocida,2009
gradjanska_vs_konstitutivni,1666
izborna_reforma,1160


In [14]:
import pandas as pd
import numpy as np

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

dfc = df_cluster.copy()

mention_cols = [f"{t}_mentioned" for t in TOPICS]

# Ako su True/False, 0/1 ili stringovi, sve prebaci u 0/1
def to_int_bool(x):
    if pd.isna(x):
        return 0
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, float)):
        return int(x == 1)
    if isinstance(x, str):
        return int(x.strip().lower() in ["true", "1", "yes", "da"])
    return 0

for col in mention_cols:
    dfc[col] = dfc[col].apply(to_int_bool)

In [15]:
cluster_topic_counts = (
    dfc
    .groupby("cluster")[mention_cols]
    .sum()
    .astype(int)
)

cluster_topic_counts

,euroatlantske_integracije_mentioned,negiranje_genocida_mentioned,gradjanska_vs_konstitutivni_mentioned,izborna_reforma_mentioned
cluster,,,,
0,1439,162,183,68
1,1075,1600,704,511
2,1619,1154,1927,684
3,0,0,0,0
4,2154,569,2478,1764


In [16]:
df_cluster["not_mentioned"] = (
    df_cluster[mention_cols].sum(axis=1) == 0
).astype(int)

not_mentioned_by_cluster = df_cluster.groupby("cluster")["not_mentioned"].sum()

not_mentioned_by_cluster

,not_mentioned
cluster,
0,1729
1,3886
2,1578
3,144
4,266
